# PND 2025 | Camada BRONZE

**Objetivo:** ler os arquivos `.txt` da landing e gravá-los em tabelas **Delta** na camada bronze,
**sem interpretar nem corrigir o conteúdo**, acrescentando metadados de rastreabilidade e um log de auditoria.

**Princípios da bronze**
- **Fidelidade à origem:** nada de conversão de tipos, filtros ou regras de negócio. Tudo entra como texto.
- **Rastreabilidade:** cada linha sabe de qual arquivo veio, quando foi ingerida e em qual lote.
- **Auditoria:** cada ingestão (sucesso ou erro) vira uma linha na tabela `bronze.ctl_ingestion_log`.
- **Reprocessável:** a landing é imutável, então a bronze pode ser refeita a qualquer momento a partir dela.

**Dois modos de leitura** (escolhidos por arquivo)
| Modo | Quando | Resultado na tabela |
|---|---|---|
| `raw` | Arquivo de **posição fixa** (ou formato desconhecido) | Uma coluna `linha_raw` com a linha inteira. O fatiamento por posição é feito na **silver**, usando o dicionário de dados |
| `delimited` | Arquivo com separador (`;`, `,`, `\|`, tab) | Uma coluna por campo, **todas em string** |

**Fluxo do notebook**
```
Preparação (1-5) → Descoberta dos arquivos (6-10) → Configuração (11-13)
   → Funções do ETL (14-20) → Pré-visualização (21) → Execução (22)
   → Auditoria e validação (23-26) → Catálogo de apoio (27-28)
```

> Execute as células **uma a uma, em ordem**. Este notebook é reexecutável: cada tabela é recriada a partir do arquivo.
> Pré-requisito: notebook da landing concluído, com os arquivos já nos volumes.

## Passo 1 | Definir o catálogo

Mesmo parâmetro da landing. O widget evita editar código para trocar de catálogo.

In [0]:
dbutils.widgets.text("catalog", "pnd2025", "Catálogo")

## Passo 2 | Ler o parâmetro e definir constantes

Guardamos o catálogo e os nomes dos schemas/volumes em variáveis, para que o resto do notebook não tenha nomes fixos espalhados.

In [0]:
CATALOG = dbutils.widgets.get("catalog").lower()

SCHEMA_LANDING = "landing"
SCHEMA_BRONZE = "bronze"

VOLUME_DADOS = "dados_txt"
VOLUME_DICIONARIOS = "dicionarios"
VOLUME_APOIO = "apoio"

print(f"Catálogo: {CATALOG}")

Catálogo: pnd2025


## Passo 3 | Selecionar o catálogo

Define o catálogo padrão da sessão. As células `%sql` mais adiante dependem disto: elas usam nomes como `bronze.ctl_ingestion_log`
sem informar o catálogo. Se o Python for reiniciado, rode este passo de novo.

In [0]:
spark.sql(f"USE CATALOG `{CATALOG}`")
display(spark.sql("SELECT current_catalog() AS catalogo_atual"))

catalogo_atual
pnd2025


## Passo 4 | Garantir que o schema bronze existe

Já foi criado no notebook da landing; `IF NOT EXISTS` só protege caso você rode este notebook isolado.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze
COMMENT 'Camada bronze - dado cru em Delta, com metadados de ingestao'

## Passo 5 | Definir os caminhos dos volumes

Mesmo padrão `/Volumes/catalogo/schema/volume/` usado na landing.

In [0]:
PATH_DADOS = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/{VOLUME_DADOS}/"
PATH_DICIONARIOS = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/{VOLUME_DICIONARIOS}/"
PATH_APOIO = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/{VOLUME_APOIO}/"

print(PATH_DADOS, PATH_DICIONARIOS, PATH_APOIO, sep="\n")

/Volumes/pnd2025/landing/dados_txt/
/Volumes/pnd2025/landing/dicionarios/
/Volumes/pnd2025/landing/apoio/


## Passo 6 | Função para listar arquivos recursivamente

`dbutils.fs.ls` não entra em subpastas. Esta função percorre tudo e devolve só arquivos. Esta célula apenas define a função.

In [0]:
def list_files(path: str):
    files = []
    for item in dbutils.fs.ls(path):
        if item.isDir():
            files.extend(list_files(item.path))
        else:
            files.append(item)
    return files

## Passo 7 | Inventariar os arquivos de dados

Lista os arquivos da landing que serão ingeridos. Dois filtros, para não ingerir nada por engano:

- `EXTENSOES`: somente `.txt`.
- `PADRAO_ARQUIVOS`: expressão regular com o nome esperado, `microdados2025_pnd_arq<número>.txt`.

A lista é ordenada em **ordem natural** (`arq1, arq2, ... arq10`). Na ordem alfabética comum, `arq10` apareceria antes de `arq2`.
`ESPERADOS` é a quantidade de arquivos prevista (13): se divergir, a célula avisa. Confira nomes e tamanhos: é o ponto de partida da auditoria.

In [0]:
import re

EXTENSOES = (".txt",)
PADRAO_ARQUIVOS = r"^microdados2025_pnd_arq\d+\.txt$"
ESPERADOS = 13


def chave_natural(f):
    # ordem natural: arq2 vem antes de arq10
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", f.name)]


arquivos = [
    f for f in list_files(PATH_DADOS)
    if f.name.lower().endswith(EXTENSOES) and re.match(PADRAO_ARQUIVOS, f.name)
]
arquivos.sort(key=chave_natural)

print(f"{len(arquivos)} arquivo(s) encontrado(s) em {PATH_DADOS}")
for f in arquivos:
    print(f"  {f.name:<40} {f.size / 1024 / 1024:>10.2f} MB")

total_mb = sum(f.size for f in arquivos) / 1024 / 1024
print(f"\nTotal: {total_mb:.1f} MB")
if len(arquivos) != ESPERADOS:
    print(f"ATENCAO: esperados {ESPERADOS} arquivos, encontrados {len(arquivos)}.")

13 arquivo(s) encontrado(s) em /Volumes/pnd2025/landing/dados_txt/
  microdados2025_pnd_arq1.txt                  354.33 MB
  microdados2025_pnd_arq2.txt                   15.38 MB
  microdados2025_pnd_arq3.txt                   14.35 MB
  microdados2025_pnd_arq4.txt                   15.10 MB
  microdados2025_pnd_arq5.txt                   15.10 MB
  microdados2025_pnd_arq6.txt                   15.12 MB
  microdados2025_pnd_arq7.txt                   15.10 MB
  microdados2025_pnd_arq8.txt                   15.10 MB
  microdados2025_pnd_arq9.txt                   15.10 MB
  microdados2025_pnd_arq10.txt                  15.10 MB
  microdados2025_pnd_arq11.txt                  15.10 MB
  microdados2025_pnd_arq12.txt                  15.10 MB
  microdados2025_pnd_arq13.txt                  15.10 MB

Total: 535.1 MB


## Passo 8 | Funções de inspeção (encoding e formato)

Antes de ler, precisamos saber **como o arquivo está codificado** e **se há separador**:

- **Encoding:** arquivos de órgãos públicos costumam vir em `ISO-8859-1` (Latin-1), não em UTF-8. Lido com o encoding errado,
  "Educação" vira "EducaÃ§Ã£o". A função tenta UTF-8; se falhar, assume `ISO-8859-1`.
- **Separador:** testa `;`, `,`, `|` e tab. Só aceita um candidato se ele aparecer **o mesmo número de vezes em todas as linhas** da amostra.
  Se nenhum passar, o arquivo é tratado como posição fixa (ou formato desconhecido) e vai pelo modo `raw`.
- **Cabeçalho:** palpite simples: a primeira linha é cabeçalho se nenhum campo dela for puramente numérico.

São heurísticas: o próximo passo mostra o resultado para você **conferir**, e o passo 12 permite corrigir.
Esta célula só define funções.

In [0]:
import codecs

TAMANHO_AMOSTRA = 262144   # bytes lidos do início de cada arquivo
CANDIDATOS_SEP = [";", ",", "|", "\t"]


def ler_amostra(path: str, nbytes: int = TAMANHO_AMOSTRA) -> bytes:
    with open(path, "rb") as f:
        return f.read(nbytes)


def detectar_encoding(raw: bytes) -> str:
    # decodificador incremental: não acusa erro se a amostra cortou um caractere ao meio
    try:
        codecs.getincrementaldecoder("utf-8")().decode(raw, final=False)
        return "UTF-8"
    except UnicodeDecodeError:
        return "ISO-8859-1"


def detectar_separador(linhas):
    melhor = None
    for c in CANDIDATOS_SEP:
        contagens = [l.count(c) for l in linhas]
        if contagens and min(contagens) > 0 and len(set(contagens)) == 1:
            if melhor is None or contagens[0] > melhor[1]:
                melhor = (c, contagens[0])
    return melhor[0] if melhor else None


def palpite_cabecalho(primeira_linha: str, sep: str) -> bool:
    campos = [c.strip().strip('"') for c in primeira_linha.split(sep)]
    return all(not c.isdigit() for c in campos)

## Passo 9 | Inspecionar cada arquivo

Para cada arquivo, lê o início, detecta encoding e separador e guarda o resultado em `deteccao`.
Também mede se **todas as linhas têm o mesmo comprimento**, um forte indício de **posição fixa**.

In [0]:
deteccao = []

for f in arquivos:
    caminho = f.path.replace("dbfs:", "")
    bruto = ler_amostra(caminho)
    enc = detectar_encoding(bruto)

    linhas = bruto.decode(enc, errors="replace").splitlines()
    if len(bruto) >= TAMANHO_AMOSTRA and linhas:
        linhas = linhas[:-1]                      # última linha da amostra pode estar cortada
    linhas = [l for l in linhas if l.strip()][:50]

    sep = detectar_separador(linhas) if linhas else None
    comprimentos = {len(l) for l in linhas}

    deteccao.append({
        "arquivo": f.name,
        "caminho": caminho,
        "encoding": enc,
        "separador": sep,
        "cabecalho_provavel": palpite_cabecalho(linhas[0], sep) if (linhas and sep) else None,
        "comprimento_fixo": (comprimentos.pop() if len(comprimentos) == 1 else None),
        "primeira_linha": (linhas[0][:120] if linhas else ""),
    })

print(f"{len(deteccao)} arquivo(s) inspecionado(s).")

13 arquivo(s) inspecionado(s).


## Passo 10 | Revisar o resultado da inspeção

**Confira com o seu conhecimento dos arquivos.** Como ler a tabela:

- `separador` preenchido → provavelmente delimitado.
- `separador` vazio e `comprimento_fixo` preenchido → provavelmente **posição fixa**.
- `separador` vazio e `comprimento_fixo` vazio → formato desconhecido; será tratado como `raw`.
- `encoding`: se tiver dúvida, compare `primeira_linha` com acentos reais.

In [0]:
import pandas as pd

display(pd.DataFrame(deteccao).drop(columns=["caminho"]))

arquivo,encoding,separador,cabecalho_provavel,comprimento_fixo,primeira_linha
microdados2025_pnd_arq1.txt,UTF-8,;,true,null,"""NU_ANO"";""CO_GRUPO"";""CO_MUNICIPIO_PROVA"";""SG_UF_MUNICIPIO_PROVA"";""TP_INSCRICAO_PND"";""IN_REAPLICACAO"";""CO_CADERNO"";""DS_VT"
microdados2025_pnd_arq2.txt,UTF-8,;,true,null,"""NU_ANO"";""CO_GRUPO"";""TP_SEXO"""
microdados2025_pnd_arq3.txt,UTF-8,;,true,null,"""NU_ANO"";""CO_GRUPO"";""NU_IDADE"""
microdados2025_pnd_arq4.txt,UTF-8,;,true,null,"""NU_ANO"";""CO_GRUPO"";""NU_QUESTAO_01"""
microdados2025_pnd_arq5.txt,UTF-8,;,true,null,"""NU_ANO"";""CO_GRUPO"";""NU_QUESTAO_02"""
microdados2025_pnd_arq6.txt,UTF-8,;,true,null,"""NU_ANO"";""CO_GRUPO"";""NU_QUESTAO_03"""
microdados2025_pnd_arq7.txt,UTF-8,;,true,null,"""NU_ANO"";""CO_GRUPO"";""NU_QUESTAO_04"""
microdados2025_pnd_arq8.txt,UTF-8,;,true,null,"""NU_ANO"";""CO_GRUPO"";""NU_QUESTAO_05"""
microdados2025_pnd_arq9.txt,UTF-8,;,true,null,"""NU_ANO"";""CO_GRUPO"";""NU_QUESTAO_06"""
microdados2025_pnd_arq10.txt,UTF-8,;,true,null,"""NU_ANO"";""CO_GRUPO"";""NU_QUESTAO_07"""


## Passo 11 | Funções de nomeação (tabelas e colunas)

Delta não aceita certos caracteres em nomes de colunas, e nomes de arquivo viram nomes de tabela.
A função `slugify` remove acentos, troca símbolos por `_` e coloca em minúsculas
(ex.: `Região do País.txt` → `regiao_do_pais`). Esta célula só define funções.

In [0]:
import re
import unicodedata


def slugify(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    texto = re.sub(r"[^0-9a-zA-Z]+", "_", texto).strip("_").lower()
    return texto or "col"


def nome_tabela(nome_arquivo: str) -> str:
    base = re.sub(r"\.[^.]+$", "", nome_arquivo)      # tira a extensão
    nome = slugify(base)
    return ("t_" + nome) if nome[0].isdigit() else nome


def nomes_colunas_unicos(nomes):
    vistos, saida = {}, []
    for i, n in enumerate(nomes, 1):
        s = slugify(n) if str(n).strip() else f"col_{i:03d}"
        if s[0].isdigit():
            s = "c_" + s
        if s in vistos:
            vistos[s] += 1
            s = f"{s}_{vistos[s]}"
        else:
            vistos[s] = 1
        saida.append(s)
    return saida

## Passo 12 | Montar a configuração de ingestão

Cria uma configuração por arquivo a partir da inspeção: `modo`, `encoding`, `separador`, `cabecalho` e tabela de destino.

**Regra automática:** separador detectado → `delimited`; caso contrário → `raw`.

**Para corrigir qualquer decisão**, preencha `OVERRIDES` (e rode a célula novamente). Exemplos:
```python
OVERRIDES = {
    "ARQUIVO_A.txt": {"mode": "raw"},                      # forçar modo raw
    "ARQUIVO_B.txt": {"encoding": "ISO-8859-1"},           # corrigir encoding
    "ARQUIVO_C.txt": {"sep": "|", "header": False},        # corrigir separador/cabeçalho
    "ARQUIVO_D.txt": {"table": "pessoas"},                 # escolher o nome da tabela
}
```

In [0]:
OVERRIDES = {
    # "NOME_DO_ARQUIVO.txt": {"mode": "raw"},
}

CONFIG = []
for d in deteccao:
    cfg = {
        "file": d["arquivo"],
        "path": d["caminho"],
        "table": nome_tabela(d["arquivo"]),
        "mode": "delimited" if d["separador"] else "raw",
        "encoding": d["encoding"],
        "sep": d["separador"],
        "header": bool(d["cabecalho_provavel"]) if d["separador"] else False,
    }
    cfg.update(OVERRIDES.get(d["arquivo"], {}))
    CONFIG.append(cfg)

display(pd.DataFrame(CONFIG).drop(columns=["path"]))

file,table,mode,encoding,sep,header
microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1,delimited,UTF-8,;,true
microdados2025_pnd_arq2.txt,microdados2025_pnd_arq2,delimited,UTF-8,;,true
microdados2025_pnd_arq3.txt,microdados2025_pnd_arq3,delimited,UTF-8,;,true
microdados2025_pnd_arq4.txt,microdados2025_pnd_arq4,delimited,UTF-8,;,true
microdados2025_pnd_arq5.txt,microdados2025_pnd_arq5,delimited,UTF-8,;,true
microdados2025_pnd_arq6.txt,microdados2025_pnd_arq6,delimited,UTF-8,;,true
microdados2025_pnd_arq7.txt,microdados2025_pnd_arq7,delimited,UTF-8,;,true
microdados2025_pnd_arq8.txt,microdados2025_pnd_arq8,delimited,UTF-8,;,true
microdados2025_pnd_arq9.txt,microdados2025_pnd_arq9,delimited,UTF-8,;,true
microdados2025_pnd_arq10.txt,microdados2025_pnd_arq10,delimited,UTF-8,;,true


## Passo 13 | Validar a configuração

Duas verificações que evitam problemas silenciosos:
1. **Nomes de tabela únicos:** dois arquivos não podem gerar a mesma tabela (um sobrescreveria o outro).
2. **Modo `delimited` exige separador.**

Se algo falhar, ajuste `OVERRIDES` no passo 12.

In [0]:
tabelas = [c["table"] for c in CONFIG]
duplicadas = {t for t in tabelas if tabelas.count(t) > 1}
assert not duplicadas, f"Nomes de tabela duplicados: {duplicadas}. Use OVERRIDES com a chave 'table'."

for c in CONFIG:
    assert c["mode"] in ("raw", "delimited"), f"Modo inválido em {c['file']}: {c['mode']}"
    if c["mode"] == "delimited":
        assert c["sep"], f"{c['file']}: modo delimited sem separador."

print(f"Configuração válida: {len(CONFIG)} arquivo(s) -> {len(set(tabelas))} tabela(s).")

Configuração válida: 13 arquivo(s) -> 13 tabela(s).


## Passo 14 | Criar a tabela de log de auditoria

Registra **cada tentativa de ingestão**: arquivo, tabela de destino, modo, linhas gravadas, status (`SUCESSO`/`ERRO`),
mensagem de erro, início, fim e duração. O prefixo `ctl_` separa tabelas de controle das tabelas de dados.

A tabela é **append-only**: o histórico de todas as execuções é preservado.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze.ctl_ingestion_log (
  batch_id        STRING    COMMENT 'Identificador do lote de execucao',
  arquivo         STRING    COMMENT 'Nome do arquivo de origem',
  caminho         STRING    COMMENT 'Caminho do arquivo no volume',
  tabela_destino  STRING    COMMENT 'Tabela bronze gravada',
  modo            STRING    COMMENT 'raw ou delimited',
  encoding        STRING    COMMENT 'Encoding usado na leitura',
  separador       STRING    COMMENT 'Separador (quando delimited)',
  linhas_gravadas BIGINT    COMMENT 'Linhas na tabela apos a gravacao',
  status          STRING    COMMENT 'SUCESSO ou ERRO',
  mensagem        STRING    COMMENT 'Detalhe do erro, se houver',
  inicio          TIMESTAMP COMMENT 'Inicio da ingestao do arquivo',
  fim             TIMESTAMP COMMENT 'Fim da ingestao do arquivo',
  duracao_seg     DOUBLE    COMMENT 'Duracao em segundos'
)
USING DELTA
COMMENT 'Log de auditoria das ingestoes da camada bronze'

## Passo 15 | Identificador do lote (`batch_id`)

Todas as linhas e todos os registros de log desta execução carregam o mesmo `batch_id`. Isso permite responder
"o que entrou em qual execução?" e comparar execuções diferentes.

In [0]:
from datetime import datetime
from uuid import uuid4

BATCH_ID = datetime.now().strftime("%Y%m%d%H%M%S") + "-" + uuid4().hex[:8]
print("BATCH_ID:", BATCH_ID)

BATCH_ID: 20261007182812-d7e9a884


## Passo 16 | Função de leitura no modo `raw`

Lê o arquivo **linha a linha**, em uma única coluna `linha_raw`.

**Detalhe técnico:** o leitor `text` do Spark não aplica a opção de encoding de forma confiável. Por isso usamos o leitor CSV
com um separador que não existe nos dados (`\u0001`) e as aspas desativadas: assim cada linha vira uma coluna só,
**respeitando o encoding** informado. Espaços no início e no fim da linha são preservados, o que é essencial para posição fixa.

A coluna oculta `_metadata` é selecionada junto, para alimentar os metadados no passo 18.

In [0]:
from pyspark.sql import functions as F


def read_raw(path: str, encoding: str):
    df = (
        spark.read.format("csv")
        .option("header", "false")
        .option("sep", "\u0001")
        .option("quote", "")
        .option("encoding", encoding)
        .option("ignoreLeadingWhiteSpace", "false")
        .option("ignoreTrailingWhiteSpace", "false")
        .load(path)
    )
    return df.select(F.col("_c0").alias("linha_raw"), "_metadata")

## Passo 17 | Função de leitura no modo `delimited`

Lê o arquivo separando os campos, mas **sem inferir tipos** (`inferSchema=false`): tudo permanece `string`.
Os nomes das colunas passam por `nomes_colunas_unicos` (sem acento, sem símbolos, sem duplicidade).
Sem cabeçalho, as colunas viram `col_001`, `col_002`, e assim por diante.

In [0]:
def read_delimited(path: str, encoding: str, sep: str, header: bool):
    df = (
        spark.read.format("csv")
        .option("header", str(header).lower())
        .option("sep", sep)
        .option("encoding", encoding)
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .load(path)
    )
    originais = df.columns
    novos = nomes_colunas_unicos(originais if header else [f"col_{i:03d}" for i in range(1, len(originais) + 1)])
    colunas = [F.col(f"`{o}`").alias(n) for o, n in zip(originais, novos)]
    return df.select(*colunas, "_metadata")

## Passo 18 | Função de metadados de rastreabilidade

Acrescenta colunas de linhagem a cada linha, todas com prefixo `_`:

| Coluna | Conteúdo |
|---|---|
| `_source_file` | Caminho completo do arquivo de origem |
| `_source_file_name` | Nome do arquivo |
| `_source_file_size` | Tamanho em bytes |
| `_source_file_modified_at` | Data de modificação do arquivo |
| `_ingestion_ts` | Momento da ingestão |
| `_batch_id` | Lote da execução (passo 15) |

In [0]:
def add_metadata(df, batch_id: str):
    return (
        df.withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_source_file_name", F.col("_metadata.file_name"))
          .withColumn("_source_file_size", F.col("_metadata.file_size"))
          .withColumn("_source_file_modified_at", F.col("_metadata.file_modification_time"))
          .withColumn("_ingestion_ts", F.current_timestamp())
          .withColumn("_batch_id", F.lit(batch_id))
          .drop("_metadata")
    )

## Passo 19 | Função de gravação do log

Insere **uma linha** em `bronze.ctl_ingestion_log` com o resultado da ingestão de um arquivo.
É chamada tanto em caso de sucesso quanto de erro.

In [0]:
LOG_SCHEMA = (
    "batch_id string, arquivo string, caminho string, tabela_destino string, modo string, "
    "encoding string, separador string, linhas_gravadas bigint, status string, mensagem string, "
    "inicio timestamp, fim timestamp, duracao_seg double"
)


def write_log(r: dict):
    row = (
        r["batch_id"], r["arquivo"], r["caminho"], r["tabela_destino"], r["modo"],
        r["encoding"], r["separador"], r["linhas_gravadas"], r["status"], r["mensagem"],
        r["inicio"], r["fim"], r["duracao_seg"],
    )
    (
        spark.createDataFrame([row], schema=LOG_SCHEMA)
        .write.mode("append")
        .saveAsTable(f"`{CATALOG}`.{SCHEMA_BRONZE}.ctl_ingestion_log")
    )

## Passo 20 | Função principal de ingestão de um arquivo

Orquestra o ETL de **um** arquivo:

1. **Extract:** lê conforme o modo (`raw` ou `delimited`).
2. **Transform:** apenas acrescenta metadados. Nenhuma regra de negócio.
3. **Load:** grava em Delta com `overwrite`. Como cada arquivo tem a sua tabela e a landing é imutável, recriar a tabela é seguro e idempotente.
4. **Audit:** conta as linhas gravadas e registra no log.

Erros são **capturados e registrados**: um arquivo com problema não interrompe a carga dos demais.
O histórico de versões de cada tabela fica disponível em `DESCRIBE HISTORY`.

In [0]:
def ingest_file(cfg: dict, batch_id: str) -> dict:
    inicio = datetime.now()
    destino = f"`{CATALOG}`.{SCHEMA_BRONZE}.`{cfg['table']}`"
    linhas, status, mensagem = None, "SUCESSO", None

    try:
        # 1) Extract
        if cfg["mode"] == "raw":
            df = read_raw(cfg["path"], cfg["encoding"])
        else:
            df = read_delimited(cfg["path"], cfg["encoding"], cfg["sep"], cfg["header"])

        # 2) Transform (somente metadados)
        df = add_metadata(df, batch_id)

        # 3) Load
        (
            df.write.format("delta")
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .saveAsTable(destino)
        )

        # 4) Audit
        linhas = spark.table(destino).count()
        origem = cfg["file"].replace("'", "''")
        spark.sql(f"COMMENT ON TABLE {destino} IS 'Bronze - origem: {origem} (modo {cfg['mode']})'")

    except Exception as e:
        status, mensagem = "ERRO", str(e)[:2000]

    fim = datetime.now()
    resultado = {
        "batch_id": batch_id, "arquivo": cfg["file"], "caminho": cfg["path"],
        "tabela_destino": cfg["table"], "modo": cfg["mode"], "encoding": cfg["encoding"],
        "separador": cfg["sep"], "linhas_gravadas": linhas, "status": status,
        "mensagem": mensagem, "inicio": inicio, "fim": fim,
        "duracao_seg": (fim - inicio).total_seconds(),
    }
    write_log(resultado)
    return resultado

## Passo 21 | Pré-visualizar a leitura de um arquivo (sem gravar)

Antes da carga completa, veja **como o arquivo será lido**. Altere `ARQUIVO_PREVIEW` para testar outros.
Nada é gravado aqui. Verifique:

- Acentos corretos? Se aparecer "Ã§", "Ã£", o encoding está errado (corrija no passo 12).
- No modo `delimited`: o número de colunas faz sentido? Os nomes estão corretos?
- No modo `raw`: a linha aparece inteira, com os espaços preservados?

In [0]:
ARQUIVO_PREVIEW = CONFIG[0]["file"]    # troque pelo nome de outro arquivo, se quiser

cfg_prev = next(c for c in CONFIG if c["file"] == ARQUIVO_PREVIEW)
print(f"Arquivo: {cfg_prev['file']} | modo: {cfg_prev['mode']} | encoding: {cfg_prev['encoding']}")

if cfg_prev["mode"] == "raw":
    df_prev = read_raw(cfg_prev["path"], cfg_prev["encoding"])
else:
    df_prev = read_delimited(cfg_prev["path"], cfg_prev["encoding"], cfg_prev["sep"], cfg_prev["header"])

df_prev = add_metadata(df_prev, "PREVIEW")
df_prev.printSchema()
display(df_prev.limit(10))

Arquivo: microdados2025_pnd_arq1.txt | modo: delimited | encoding: UTF-8
root
 |-- nu_ano: integer (nullable = true)
 |-- co_grupo: integer (nullable = true)
 |-- co_municipio_prova: integer (nullable = true)
 |-- sg_uf_municipio_prova: string (nullable = true)
 |-- tp_inscricao_pnd: integer (nullable = true)
 |-- in_reaplicacao: integer (nullable = true)
 |-- co_caderno: integer (nullable = true)
 |-- ds_vt_gab_obj: string (nullable = true)
 |-- ds_vt_esc_obj: string (nullable = true)
 |-- ds_vt_ace_obj: string (nullable = true)
 |-- tp_pres: integer (nullable = true)
 |-- tp_sit_disc: integer (nullable = true)
 |-- proficiencia: string (nullable = true)
 |-- nt_obj: string (nullable = true)
 |-- nt_dis: string (nullable = true)
 |-- nt_ger: string (nullable = true)
 |-- qt_acertos: string (nullable = true)
 |-- co_rs_i1: string (nullable = true)
 |-- co_rs_i2: string (nullable = true)
 |-- co_rs_i3: string (nullable = true)
 |-- co_rs_i4: string (nullable = true)
 |-- co_rs_i5: strin

nu_ano,co_grupo,co_municipio_prova,sg_uf_municipio_prova,tp_inscricao_pnd,in_reaplicacao,co_caderno,ds_vt_gab_obj,ds_vt_esc_obj,ds_vt_ace_obj,tp_pres,tp_sit_disc,proficiencia,nt_obj,nt_dis,nt_ger,qt_acertos,co_rs_i1,co_rs_i2,co_rs_i3,co_rs_i4,co_rs_i5,co_rs_i6,co_rs_i7,co_rs_i8,co_rs_i9,_source_file,_source_file_name,_source_file_size,_source_file_modified_at,_ingestion_ts,_batch_id
2025,702,1100205,RO,2,0,3,CACABACDDBCADABDDBADCCCCAACDBBBDCDDBCDCABDBADDCAACACBBACDDBBDCAACACAACBCADBDDBBA,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,333,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:34:08.965Z,PREVIEW
2025,702,1100205,RO,2,0,3,CACABACDDBCADABDDBADCCCCAACDBBBDCDDBCDCABDBADDCAACACBBACDDBBDCAACACAACBCADBDDBBA,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:34:08.965Z,PREVIEW
2025,702,1200203,AC,2,0,3,CACABACDDBCADABDDBADCCCCAACDBBBDCDDBCDCABDBADDCAACACBBACDDBBDCAACACAACBCADBDDBBA,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:34:08.965Z,PREVIEW
2025,702,1501402,PA,2,0,2,CDBADBCADBBBDDDADCCABAACCCCACABCDCABDBDCDDACACBBACBADDCACAACACAADDBBDDDBBACBCADB,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,333,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:34:08.965Z,PREVIEW
2025,702,1702208,TO,2,0,4,DBBCAACDCCCDBCACACDCADABDDBABABACDDCACAACADBDCABCBDCDDACACBDDBBABDBACABDCDDBAACB,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:34:08.965Z,PREVIEW
2025,702,1702406,TO,2,0,2,CDBADBCADBBBDDDADCCABAACCCCACABCDCABDBDCDDACACBBACBADDCACAACACAADDBBDDDBBACBCADB,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:34:08.965Z,PREVIEW
2025,702,1705508,TO,2,0,3,CACABACDDBCADABDDBADCCCCAACDBBBDCDDBCDCABDBADDCAACACBBACDDBBDCAACACAACBCADBDDBBA,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:34:08.965Z,PREVIEW
2025,702,2208403,PI,2,0,1,DBCDCABDDDADCBACAACCCCACADBBBABDBDCDDACACBBACBADDCACAACACADCAADDBBDDDBBACBCADBBC,................................................................................,00800000000000800

## Passo 22 | Executar a ingestão de todos os arquivos

Percorre `CONFIG`, chama `ingest_file` para cada arquivo e mostra o resultado. Pode levar alguns minutos conforme o volume.
Se algum arquivo falhar, ele aparece com `status = ERRO` e a mensagem; os demais seguem normalmente.

In [0]:
resultados = []
for i, cfg in enumerate(CONFIG, 1):
    r = ingest_file(cfg, BATCH_ID)
    resultados.append(r)
    print(f"[{i}/{len(CONFIG)}] {r['arquivo']} -> {r['tabela_destino']}: {r['status']} "
          f"({r['linhas_gravadas']} linhas, {r['duracao_seg']:.1f}s)")

ok = sum(r["status"] == "SUCESSO" for r in resultados)
print(f"\nConcluído: {ok} sucesso(s), {len(resultados) - ok} erro(s). Lote: {BATCH_ID}")

[1/13] microdados2025_pnd_arq1.txt -> microdados2025_pnd_arq1: SUCESSO (1087359 linhas, 13.2s)
[2/13] microdados2025_pnd_arq2.txt -> microdados2025_pnd_arq2: SUCESSO (1087359 linhas, 7.2s)
[3/13] microdados2025_pnd_arq3.txt -> microdados2025_pnd_arq3: SUCESSO (1087359 linhas, 6.8s)
[4/13] microdados2025_pnd_arq4.txt -> microdados2025_pnd_arq4: SUCESSO (1087359 linhas, 7.0s)
[5/13] microdados2025_pnd_arq5.txt -> microdados2025_pnd_arq5: SUCESSO (1087359 linhas, 6.3s)
[6/13] microdados2025_pnd_arq6.txt -> microdados2025_pnd_arq6: SUCESSO (1087359 linhas, 6.4s)
[7/13] microdados2025_pnd_arq7.txt -> microdados2025_pnd_arq7: SUCESSO (1087359 linhas, 6.2s)
[8/13] microdados2025_pnd_arq8.txt -> microdados2025_pnd_arq8: SUCESSO (1087359 linhas, 6.2s)
[9/13] microdados2025_pnd_arq9.txt -> microdados2025_pnd_arq9: SUCESSO (1087359 linhas, 6.3s)
[10/13] microdados2025_pnd_arq10.txt -> microdados2025_pnd_arq10: SUCESSO (1087359 linhas, 6.2s)
[11/13] microdados2025_pnd_arq11.txt -> microdados2025_p

## Passo 23 | Consultar o log de auditoria deste lote

Leitura da tabela de controle, filtrando pelo lote atual. Em caso de `ERRO`, a coluna `mensagem` traz o detalhe.

In [0]:
display(
    spark.table(f"`{CATALOG}`.{SCHEMA_BRONZE}.ctl_ingestion_log")
    .where(F.col("batch_id") == BATCH_ID)
    .orderBy("inicio")
)

batch_id,arquivo,caminho,tabela_destino,modo,encoding,separador,linhas_gravadas,status,mensagem,inicio,fim,duracao_seg
20261007182812-d7e9a884,microdados2025_pnd_arq1.txt,/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1,delimited,UTF-8,;,1087359,SUCESSO,null,2026-10-07T18:35:20.843Z,2026-10-07T18:35:34.046Z,13.20233
20261007182812-d7e9a884,microdados2025_pnd_arq2.txt,/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq2.txt,microdados2025_pnd_arq2,delimited,UTF-8,;,1087359,SUCESSO,null,2026-10-07T18:35:36.203Z,2026-10-07T18:35:43.450Z,7.247541
20261007182812-d7e9a884,microdados2025_pnd_arq3.txt,/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq3.txt,microdados2025_pnd_arq3,delimited,UTF-8,;,1087359,SUCESSO,null,2026-10-07T18:35:45.092Z,2026-10-07T18:35:51.897Z,6.804424
20261007182812-d7e9a884,microdados2025_pnd_arq4.txt,/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq4.txt,microdados2025_pnd_arq4,delimited,UTF-8,;,1087359,SUCESSO,null,2026-10-07T18:35:53.368Z,2026-10-07T18:36:00.334Z,6.965811
20261007182812-d7e9a884,microdados2025_pnd_arq5.txt,/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq5.txt,microdados2025_pnd_arq5,delimited,UTF-8,;,1087359,SUCESSO,null,2026-10-07T18:36:01.669Z,2026-10-07T18:36:08.016Z,6.347185
20261007182812-d7e9a884,microdados2025_pnd_arq6.txt,/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq6.txt,microdados2025_pnd_arq6,delimited,UTF-8,;,1087359,SUCESSO,null,2026-10-07T18:36:09.347Z,2026-10-07T18:36:15.783Z,6.436433
20261007182812-d7e9a884,microdados2025_pnd_arq7.txt,/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq7.txt,microdados2025_pnd_arq7,delimited,UTF-8,;,1087359,SUCESSO,null,2026-10-07T18:36:17.101Z,2026-10-07T18:36:23.300Z,6.199125
20261007182812-d7e9a884,microdados2025_pnd_arq8.txt,/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq8.txt,microdados2025_pnd_arq8,delimited,UTF-8,;,1087359,SUCESSO,null,2026-10-07T18:36:24.567Z,2026-10-07T18:36:30.815Z,6.247577
20261007182812-d7e9a884,microdados2025_pnd_arq9.txt,/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq9.txt,microdados2025_pnd_arq9,delimited,UTF-8,;,1087359,SUCESSO,null,2026-10-07T18:36:32.071Z,2026-10-07T18:36:38.331Z,6.260729
20261007182812-d7e9a884,microdados2025_pnd_arq10.txt,/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq10.txt,microdados2025_pnd_arq10,delimited,UTF-8,;,1087359,SUCESSO,null,2026-10-07T18:36:39.593Z,2026-10-07T18:36:45.811Z,6.217518


## Passo 24 | Histórico completo de execuções

Em SQL: todas as execuções já registradas, da mais recente para a mais antiga. Útil para comparar lotes ao longo do tempo.

In [0]:
%sql
SELECT batch_id, status, COUNT(*) AS arquivos, SUM(linhas_gravadas) AS total_linhas,
       MIN(inicio) AS inicio_lote, ROUND(SUM(duracao_seg), 1) AS duracao_total_seg
FROM bronze.ctl_ingestion_log
GROUP BY batch_id, status
ORDER BY inicio_lote DESC

batch_id,status,arquivos,total_linhas,inicio_lote,duracao_total_seg
20261007182812-d7e9a884,SUCESSO,13,14135667,2026-10-07T18:35:20.843Z,90.8


## Passo 25 | Validar contagem: origem x bronze

Confronta, para cada tabela, o **número de linhas do arquivo de origem** com o **número de linhas gravadas**:

- `raw`: esperado = linhas do arquivo.
- `delimited` com cabeçalho: esperado = linhas do arquivo **menos 1**.

Resultado `DIVERGENTE` merece investigação. Uma causa comum são **linhas em branco** no arquivo, que o leitor CSV pode descartar
(o log do passo 23 ajuda a entender o caso). A contagem da origem usa o leitor `text`, que não depende do encoding.

In [0]:
validacao = []
for cfg in CONFIG:
    destino = f"`{CATALOG}`.{SCHEMA_BRONZE}.`{cfg['table']}`"
    try:
        linhas_origem = spark.read.text(cfg["path"]).count()
        esperado = linhas_origem - (1 if (cfg["mode"] == "delimited" and cfg["header"]) else 0)
        linhas_tabela = spark.table(destino).count()
        situacao = "OK" if esperado == linhas_tabela else "DIVERGENTE"
    except Exception as e:
        esperado, linhas_tabela, situacao = None, None, f"ERRO: {str(e)[:80]}"

    validacao.append({
        "arquivo": cfg["file"], "tabela": cfg["table"],
        "linhas_esperadas": esperado, "linhas_bronze": linhas_tabela, "situacao": situacao,
    })

display(pd.DataFrame(validacao))

arquivo,tabela,linhas_esperadas,linhas_bronze,situacao
microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1,1087359,1087359,OK
microdados2025_pnd_arq2.txt,microdados2025_pnd_arq2,1087359,1087359,OK
microdados2025_pnd_arq3.txt,microdados2025_pnd_arq3,1087359,1087359,OK
microdados2025_pnd_arq4.txt,microdados2025_pnd_arq4,1087359,1087359,OK
microdados2025_pnd_arq5.txt,microdados2025_pnd_arq5,1087359,1087359,OK
microdados2025_pnd_arq6.txt,microdados2025_pnd_arq6,1087359,1087359,OK
microdados2025_pnd_arq7.txt,microdados2025_pnd_arq7,1087359,1087359,OK
microdados2025_pnd_arq8.txt,microdados2025_pnd_arq8,1087359,1087359,OK
microdados2025_pnd_arq9.txt,microdados2025_pnd_arq9,1087359,1087359,OK
microdados2025_pnd_arq10.txt,microdados2025_pnd_arq10,1087359,1087359,OK


## Passo 26 | Conferir as tabelas criadas na bronze

Lista as tabelas do schema. Devem aparecer as tabelas de dados (uma por arquivo) e `ctl_ingestion_log`.

In [0]:
%sql
SHOW TABLES IN bronze

database,tableName,isTemporary
bronze,ctl_ingestion_log,false
bronze,microdados2025_pnd_arq1,false
bronze,microdados2025_pnd_arq10,false
bronze,microdados2025_pnd_arq11,false
bronze,microdados2025_pnd_arq12,false
bronze,microdados2025_pnd_arq13,false
bronze,microdados2025_pnd_arq2,false
bronze,microdados2025_pnd_arq3,false
bronze,microdados2025_pnd_arq4,false
bronze,microdados2025_pnd_arq5,false


## Passo 27 | Catalogar dicionários e arquivos de apoio

Dicionários e arquivos de apoio normalmente são `xlsx`, `pdf` ou `docx`, que não viram tabelas de dados.
Mesmo assim, registramos **o que existe** (nome, caminho, tamanho, data) em uma tabela de controle, para auditoria e
para referenciar o dicionário correto quando construirmos a silver.

Se um dicionário for um arquivo texto tabular (ex.: `.csv`), podemos ingeri-lo como os demais, adicionando-o ao passo 7.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

catalogo_rows = []
for volume, path in [(VOLUME_DICIONARIOS, PATH_DICIONARIOS), (VOLUME_APOIO, PATH_APOIO)]:
    for f in list_files(path):
        mod_ms = getattr(f, "modificationTime", None)
        catalogo_rows.append((
            volume,
            f.name,
            f.path.replace("dbfs:", ""),
            f.size,
            datetime.fromtimestamp(mod_ms / 1000) if mod_ms else None,
            datetime.now(),
            BATCH_ID,
        ))

schema_catalogo = StructType([
    StructField("volume", StringType()),
    StructField("arquivo", StringType()),
    StructField("caminho", StringType()),
    StructField("tamanho_bytes", LongType()),
    StructField("modificado_em", TimestampType()),
    StructField("catalogado_em", TimestampType()),
    StructField("batch_id", StringType()),
])

df_catalogo = spark.createDataFrame(catalogo_rows, schema=schema_catalogo)
print(f"{df_catalogo.count()} arquivo(s) de apoio encontrado(s).")

6 arquivo(s) de apoio encontrado(s).


## Passo 28 | Gravar o catálogo de apoio

Grava em `bronze.ctl_arquivos_apoio` e exibe o resultado. A tabela é recriada a cada execução, refletindo o conteúdo atual dos volumes.

In [0]:
(
    df_catalogo.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{CATALOG}`.{SCHEMA_BRONZE}.ctl_arquivos_apoio")
)

spark.sql(f"COMMENT ON TABLE `{CATALOG}`.{SCHEMA_BRONZE}.ctl_arquivos_apoio IS 'Catalogo de dicionarios e arquivos de apoio da landing'")
display(spark.table(f"`{CATALOG}`.{SCHEMA_BRONZE}.ctl_arquivos_apoio").orderBy("volume", "arquivo"))

volume,arquivo,caminho,tamanho_bytes,modificado_em,catalogado_em,batch_id
apoio,Manual do usuário_microdados_PND_2025 (09-09-2026).pdf,/Volumes/pnd2025/landing/apoio/Manual do usuário_microdados_PND_2025 (09-09-2026).pdf,305930,2026-10-07T12:59:22.000Z,2026-10-07T18:41:47.017Z,20261007182812-d7e9a884
apoio,QUESTIONARIO_CONTEXTUAL_PND.pdf,/Volumes/pnd2025/landing/apoio/QUESTIONARIO_CONTEXTUAL_PND.pdf,220611,2026-10-07T12:59:22.000Z,2026-10-07T18:41:47.017Z,20261007182812-d7e9a884
dicionarios,Dicionário arquivos e variáveis PND 2025 (09-09-2029).ods,/Volumes/pnd2025/landing/dicionarios/Dicionário arquivos e variáveis PND 2025 (09-09-2029).ods,126293,2026-10-07T12:59:15.000Z,2026-10-07T18:41:46.408Z,20261007182812-d7e9a884
dicionarios,Dicionário arquivos e variáveis PND 2025 (09-09-2029).xlsx,/Volumes/pnd2025/landing/dicionarios/Dicionário arquivos e variáveis PND 2025 (09-09-2029).xlsx,182512,2026-10-07T12:59:15.000Z,2026-10-07T18:41:46.408Z,20261007182812-d7e9a884
dicionarios,microdados2025_parametros_itens.ods,/Volumes/pnd2025/landing/dicionarios/microdados2025_parametros_itens.ods,154170,2026-10-07T12:59:49.000Z,2026-10-07T18:41:46.408Z,20261007182812-d7e9a884
dicionarios,microdados2025_parametros_itens.xlsx,/Volumes/pnd2025/landing/dicionarios/microdados2025_parametros_itens.xlsx,133320,2026-10-07T12:59:49.000Z,2026-10-07T18:41:46.408Z,20261007182812-d7e9a884


## Próximo passo

`02_silver/`: tipagem, fatiamento por posição fixa (para tabelas em modo `raw`, usando o dicionário de dados),
padronização e regras de qualidade.

**Para planejar a silver, tenha à mão:** o dicionário de variáveis de cada arquivo (nome, posição inicial, tamanho e tipo)
e a lista de códigos/categorias.